# Catching and Tracing Exceptions

CSC-239 · Module 7 · Lesson 1 of 4

You have written methods that return useful results. Now you will trace what happens when a method cannot finish and choose where the program should handle that failure.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Trace which statements run after an arithmetic failure inside a loop.
- Catch the specific failure and use a method-call trace to locate its source.


## Why This Matters

A packing calculator processes several requested group sizes. A request for zero groups cannot produce an integer share, but the next request can still be useful.


## Check Your Starting Point

Explain integer division, a static method call, and the order of statements in a for loop. Recall that an object has a type and methods, and that an array provides indexed access to its elements.

**My explanation:**


## Concept

### Recognize an exception

An **exception object** describes a failure during execution. Its type tells you the kind of failure, and its message gives more detail. For example, integer division by zero raises ArithmeticException. This rule applies to integer division; floating-point division by zero follows different rules.

A missing semicolon is a compiler error. The compiler rejects that source before it can run. An exception occurs while otherwise accepted code is running.

### Leave unfinished work

**Abrupt completion** means control leaves an operation before its remaining statements finish. Java does not invent a division result and continue the next statement inside the failed operation.

A **try block** surrounds work whose exceptions you want to handle. A **catch block** receives a matching exception and provides a response. It is also called an exception handler. The variable in the catch header refers to that actual exception object.

```java
try {
    int share = 12 / 0;
    System.out.println("Share: " + share);
    System.out.println("Calculated");
} catch (ArithmeticException problem) {
    System.out.println("Cannot divide by zero.");
}
System.out.println("After handling");
```

This prints Cannot divide by zero. and then After handling. The calculation fails, so neither statement after it inside try runs. Java enters the matching catch, finishes that response, and then continues after the whole try/catch statement. It does not resume at the failed division or return to the skipped print statements.

Catch the failure you are prepared to handle. An empty catch silently discards useful information. Catching every possible failure also makes it harder to distinguish an expected input problem from a programming mistake.

### Decide how much work may be skipped

Where you put the handler changes which work can continue. Put try/catch inside a loop when each input should have its own chance:

```java
int[] divisors = {2, 0, 3};
for (int divisor : divisors) {
    try {
        System.out.println(6 / divisor);
    } catch (ArithmeticException problem) {
        System.out.println("Skipped one input");
    }
}
System.out.println("Finished");
```

This prints 3, Skipped one input, 2, and Finished on separate lines. The catch finishes the failed iteration's response. The loop can then move to its next input.

Compare that with one handler around the entire loop:

```java
int[] divisors = {2, 0, 3};
try {
    for (int divisor : divisors) {
        System.out.println(6 / divisor);
    }
} catch (ArithmeticException problem) {
    System.out.println("Left the loop");
}
System.out.println("Finished");
```

This prints 3, Left the loop, and Finished. The failure exits the loop while Java looks for the outside handler. That loop does not restart, so the final divisor is never processed.

### Follow a failure across method calls

A **stack trace** records the active chain of method calls associated with an exception. Each entry is a StackTraceElement, an object describing one recorded call. The top entry normally points to the operation where this exception originated. Following entries show callers.

**Stack unwinding** means leaving unfinished calls while searching outward for a matching catch. Consider a method that calls another method that divides by zero:

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
try {
    RatioTools.quote();
    System.out.println("Quote completed");
} catch (ArithmeticException problem) {
    System.out.println(problem.getMessage());
    System.out.println(problem.getStackTrace()[0].getMethodName());
    System.out.println(problem.getStackTrace()[1].getMethodName());
}
```

In this workspace, the output is / by zero, share, and quote. The division fails inside share. There is no handler there, so Java leaves share without returning a result. It then leaves quote and reaches the caller's catch. Quote completed is skipped.

The getMessage method returns the exception's explanation. The getStackTrace method returns an array of recorded calls. Index 0 selects the top entry; getMethodName reads the method name from that entry. Here we know the first two entries come from our two methods. A general tool should check the array length before assuming an entry exists.

You may also call problem.printStackTrace() inside a handler to display the complete trace. A notebook trace includes extra methods generated by IJava. Their file names and line numbers can change after editing or rerunning cells. Begin with your own method names and the failing operation instead of memorizing the entire generated trace. Exact message wording can also depend on the runtime; this lesson's small trace was checked in the course workspace.


## Video Demonstration

Watch where the calculation stops for a zero group count. Before the command runs, predict which status lines will be skipped and which later inputs will still be processed.

<video controls preload="metadata" width="960">
  <source src="media/01_catching_and_tracing_exceptions/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/01_catching_and_tracing_exceptions/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the catching and tracing exceptions demonstration transcript](media/01_catching_and_tracing_exceptions/transcript.md).


## Worked Example

**Subgoal 1: separate the calculation.** RatioTools.share performs integer division.

**Subgoal 2: handle each input.** Keep try/catch inside the loop so a failed input has a local response.

**Subgoal 3: mark progress.** Print Calculated only after success, Next input after either path, and Done after the loop.


In [ ]:
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {3, 0, 4};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");


Expected output:

```text
Share: 4
Calculated
Next input
Cannot divide by zero.
Next input
Share: 3
Calculated
Next input
Done
```

Three groups receive four items each. Zero groups raises ArithmeticException before either successful-status line can finish, so the handler prints its message. Four groups then receive three items each. The per-input handler lets the loop reach that final valid request.


## Predict, Run, Trace, and Explain

### Predict a failure before two valid inputs

Read the complete program before running it. Predict every output line in order for groupCounts {0, 6, 2}. For each input, decide whether share returns a result, whether Calculated runs, and whether Next input runs. Explain whether any part of the Share line prints when its calculation fails. Record your prediction before running the next cell or opening the answer.

My predicted complete output:

For each input, does share return?

Which inputs reach Calculated?

Which inputs reach Next input?

What happens to the whole Share print when the call fails?

Why the first failure does or does not prevent the later inputs:


In [ ]:
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");


Run the complete cell once. Keep your original prediction and compare every line with the actual output. Identify the first difference and explain it using the failed call, the skipped statements or the position of the catch. Run the whole cell for each attempt so its array and method definitions are present.

My original prediction:

My actual complete output:

The first matching or differing line:

The statement or handler position that explains a difference:

My corrected explanation after running:

### Trace skipped statements and unfinished calls

Complete the table for the prediction program. Separate a returned value from an exception and identify the type received by catch. Explain where execution continues after that catch finishes. Then use the two complete method-call programs below to trace stack unwinding: which unfinished methods are left while Java searches for the handler, and which recorded calls appear first in the exception’s stack trace.

| Input groups | share result or failure | Share line printed? | Calculated printed? | Catch runs? | Next input printed? |
|---|---|---|---|---|---|
| 0 | | | | | |
| 6 | | | | | |
| 2 | | | | | |


The type received by catch:

What problem refers to:

The first statement reached after the handler:

Why handling does not resume the failed calculation:

How my table explains the observed output:

<details>
<summary>Show answer</summary>

The first input is 0. Integer division fails inside share before it returns a result. The surrounding println cannot finish its argument, so it prints no Share line. Calculated is also skipped. The matching catch prints Cannot divide by zero. and then control reaches Next input. With 6 groups, share returns 2; with 2 groups, it returns 6. Both successful iterations print Share, Calculated and Next input in that order. The catch is inside the loop, so the first failure does not discard either later input. Done prints once after all three iterations. Handling the exception does not resume the failed division or the skipped statements inside try. The exception type named by the handler is ArithmeticException. The catch variable problem refers to the exception object raised during this failed division; the fixed printed sentence is the program’s chosen response. It is not a replacement numeric result. In this loop, the failed share call leaves the current print unfinished and Java finds the catch for that iteration. The separate method-call check below makes the unfinished calls and their recorded order visible.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Cannot divide by zero.
Next input
Share: 2
Calculated
Next input
Share: 6
Calculated
Next input
Done
```

Common error: Printing Share: 0 for a failed division. Printing Calculated after the failed call. Skipping the two later inputs even though the handler is inside the loop. Using the earlier worked example’s group counts or output.

</details>


### Follow a failure out of two methods

Predict the complete output of each program before running it. The first calls quote with 0; the second changes only that argument to 3. Trace entry into quote and share, and mark which Leave messages and returns are reached. A StackTraceElement is one recorded method-call entry. Here getStackTrace returns an array of those entries, and getMethodName reads a method’s name. In the failure case, identify the exact exception message and the first two recorded method names. Explain why those two names are reported in a different order from the Enter messages. Then run both complete programs, compare your predictions and explain which statements the search for a handler skipped. The first two trace entries in this specific program belong to share and its caller quote; later entries belong to the notebook environment and are not part of the output comparison.

My predicted failure output:

My predicted success output:

My actual failure output:

My actual success output:

Order of the Enter messages:

Leave messages and returns skipped after failure:

The actual exception type and message:

The first two recorded method names and why that order differs:

Where the matching handler is located:

Why After handling appears on both paths:

My post-run explanation of stack unwinding:


**First program:**


In [ ]:
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(0);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");


**Comparison program:**


In [ ]:
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(3);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

The caller first prints Start request. quote prints Enter quote and calls share, which prints Enter share. Division by 0 then raises ArithmeticException. The remaining Leave share message and return are skipped; quote also cannot finish its call, so Leave quote and its return are skipped. The caller cannot initialize result or print Quote. Java finds the caller’s matching catch. problem refers to the actual exception object; getMessage returns / by zero in the selected Workspace runtime. getStackTrace returns recorded entries with share first and quote next: the failing method followed by its caller. That recorded order is opposite the two Enter messages, which showed calls being entered. The handler does not resume either method. After handling runs after the whole try/catch. In the comparison, 18 / 3 returns 6, both Leave messages run in return order, Quote: 6 prints, and the catch body is skipped.

```java
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(0);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");
```

Expected output:

```text
Start request
Enter quote
Enter share
Problem: / by zero
Top method: share
Caller method: quote
After handling
```

Common error: Reading the Enter order as the exception trace order. Printing Leave share or Leave quote after the division fails. Assuming a matching catch repairs and resumes the unfinished methods. Comparing generated notebook file names or line numbers instead of these two method names. Assuming the label After handling means the catch ran on the successful path.

**Check case 2.** A valid divisor allows both method calls to return. Both Leave messages and Quote appear; the handler’s message and trace lines do not. After handling still runs because it follows the whole try/catch.

```java
class TracePacking {
    public static int share(int total, int groups) {
        System.out.println("Enter share");
        int result = total / groups;
        System.out.println("Leave share");
        return result;
    }
    public static int quote(int groups) {
        System.out.println("Enter quote");
        int result = share(18, groups);
        System.out.println("Leave quote");
        return result;
    }
}
try {
    System.out.println("Start request");
    int result = TracePacking.quote(3);
    System.out.println("Quote: " + result);
} catch (ArithmeticException problem) {
    System.out.println("Problem: " + problem.getMessage());
    StackTraceElement[] calls = problem.getStackTrace();
    System.out.println("Top method: " + calls[0].getMethodName());
    System.out.println("Caller method: " + calls[1].getMethodName());
}
System.out.println("After handling");
```

Expected output:

```text
Start request
Enter quote
Enter share
Leave share
Leave quote
Quote: 6
After handling
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete a handler for each box count

The displayed draft is incomplete and for reading only. Copy it into the empty work cell. Replace PROTECTED_WORK, HANDLE and FAILURE using `try`, `catch` and `ArithmeticException`, each once. Keep every other statement and input unchanged. Predict all lines, run the completed program, and explain why Ready is printed for only one input while Checked appears for both. Identify the operation that fails and the exception object received by the handler.

This sample is for repair:

```java
class PackShare {
    public static int perBox(int parcels, int boxes) {
        return parcels / boxes;
    }
}
int[] boxCounts = {0, 3};
for (int boxes : boxCounts) {
    PROTECTED_WORK {
        int each = PackShare.perBox(9, boxes);
        System.out.println("Each: " + each);
        System.out.println("Ready");
    } HANDLE (FAILURE problem) {
        System.out.println("Needs a box count.");
    }
    System.out.println("Checked");
}
System.out.println("Complete");
```


My three replacements:

My predicted complete output:

My actual complete output:

The failing method and operation:

What problem receives:

Why Ready and Checked have different counts:

My post-run explanation:

<details>
<summary>Show answer</summary>

PROTECTED_WORK is try, HANDLE is catch and FAILURE is ArithmeticException. For box count 0, the call to perBox cannot return a result, so each is not initialized and neither Each nor Ready is printed. The catch prints Needs a box count. Checked runs after that handler. For box count 3, integer division returns 3, so Each: 3 and Ready print; catch is skipped and Checked still prints. Complete appears once after the loop. The catch is inside the loop and receives the exception object for its failed calculation.

```java
class PackShare {
    public static int perBox(int parcels, int boxes) {
        return parcels / boxes;
    }
}
int[] boxCounts = {0, 3};
for (int boxes : boxCounts) {
    try {
        int each = PackShare.perBox(9, boxes);
        System.out.println("Each: " + each);
        System.out.println("Ready");
    } catch (ArithmeticException problem) {
        System.out.println("Needs a box count.");
    }
    System.out.println("Checked");
}
System.out.println("Complete");
```

Expected output:

```text
Needs a box count.
Checked
Each: 3
Ready
Checked
Complete
```

Common error: Replacing catch with another loop. Placing Ready after the catch so it falsely reports success after failure. Moving Checked into the successful try body. Changing the provided inputs to avoid handling the failing case.

</details>


### Report the failed input and its actual message

Keep RatioTools, groupCounts {0, 6, 2}, the loop and all successful/progress messages unchanged. Replace only the catch body’s print with `System.out.println("Groups " + groups + ": " + problem.getMessage());`. Predict and run the complete program. Explain which part of the diagnostic comes from the input and which part comes from the exception object. Then change only the array initializer to {2, 0, 0}, predict and run again, and explain whether both failed inputs get their own response. Restore {0, 6, 2} afterward.


In [ ]:
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");


My changed catch statement:

My predicted and actual output for {0, 6, 2}:

Which text comes from groups and which comes from problem:

My predicted and actual output for {2, 0, 0}:

Why each zero has its own response and Next input:

My corrected explanation and restored output:

<details>
<summary>Show answer</summary>

The handler now combines the current input groups with the actual exception message. For zero groups it prints Groups 0: / by zero in this Workspace runtime. That message does not provide a division result, so there is still no Share or Calculated line for that input. All Next input lines remain because their statement follows try/catch inside the loop. The valid inputs still produce shares 2 and 6. With {2, 0, 0}, the first calculation returns 6 and each later zero produces its own caught exception and diagnostic. The repeated failure does not stop the loop or remove the final Done.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {0, 6, 2};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Groups " + groups + ": " + problem.getMessage());
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Groups 0: / by zero
Next input
Share: 2
Calculated
Next input
Share: 6
Calculated
Next input
Done
```

Common error: Using the fixed friendly sentence while claiming to display the exception’s message. Moving Next input into catch and losing it on successful inputs. Treating the diagnostic as a successful calculated value. Handling only the first of two failed inputs.

**Additional test: `One valid input followed by two zero group counts`.** The per-input handler runs twice. Next input runs three times, and Done still runs once after the loop.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {2, 0, 0};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Groups " + groups + ": " + problem.getMessage());
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Share: 6
Calculated
Next input
Groups 0: / by zero
Next input
Groups 0: / by zero
Next input
Done
```

</details>


### Repair a handler that exits the whole loop

The displayed draft catches the division failure, but its handler surrounds the whole loop. For {6, 0, 3}, first predict which input is never processed and which Next input messages are missing. Keep the draft in Markdown. Copy the complete program into the empty work cell and put try/catch inside the loop so each input has its own handler. Place Next input after that try/catch but still inside the loop; keep Done after the loop. Preserve RatioTools, all inputs and every message. Predict and run the repaired program. Explain why the outside catch could handle the exception yet still fail this task’s requirement to continue with later inputs.

This sample is for repair:

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {6, 0, 3};
try {
    for (int groups : groupCounts) {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
        System.out.println("Next input");
    }
} catch (ArithmeticException problem) {
    System.out.println("Cannot divide by zero.");
}
System.out.println("Done");
```


My predicted output of the faulty draft:

The input never reached by that draft:

The missing Next input messages and their cause:

My repaired handler and progress placement:

My predicted complete repaired output:

My actual repaired output:

Why catching a failure does not restart an exited loop:

My post-run explanation:

<details>
<summary>Show answer</summary>

With the handler outside, the first input 6 returns share 2 and prints its two progress messages. The zero input causes the method call and then the whole loop to end abruptly while Java searches for the outside catch. That input never reaches Next input, and the later input 3 is not visited. After the outside catch, control reaches Done; it does not reenter the loop. The repair moves the handler inside each iteration and places Next input after that handler. The zero input now receives its response and progress marker, and the later input 3 returns share 4. No method or input change is needed.

```java
class RatioTools {
    public static int share(int total, int groups) {
        return total / groups;
    }
    public static int quote() {
        return share(12, 0);
    }
}
int[] groupCounts = {6, 0, 3};
for (int groups : groupCounts) {
    try {
        System.out.println("Share: " + RatioTools.share(12, groups));
        System.out.println("Calculated");
    } catch (ArithmeticException problem) {
        System.out.println("Cannot divide by zero.");
    }
    System.out.println("Next input");
}
System.out.println("Done");
```

Expected output:

```text
Share: 2
Calculated
Next input
Cannot divide by zero.
Next input
Share: 4
Calculated
Next input
Done
```

Common error: Changing zero to a valid number instead of repairing handler placement. Keeping the catch outside and expecting the loop to restart afterward. Leaving Next input inside try so it is still skipped on failure. Printing Calculated from catch and falsely reporting success.

</details>


## Independent Practice

### Build a packing calculation that continues after failure

Write the complete `ParcelMath` class and caller in the work cell. Define `public static int perContainer(int parcels, int containers)` to return `parcels / containers` using integer division. Use an enhanced for loop over `int[] containerCounts = {2, 0, 5};` and try the calculation for ten parcels inside each iteration. On success, print `Per container: ` plus the result, then `Packed`. Catch `ArithmeticException` to print `Choose a nonzero container count.` Print `Next input` after either path inside the loop and `Finished` once after the loop. Predict the complete output before running. Explain which statements are skipped for zero and why the final valid input is still processed. Test empty input, an initial zero followed by a valid count, and a final zero. Explain why a catch around the entire loop would skip later inputs after a failure.

My calculation method and handler placement:

My predicted complete output:

My actual complete output:

Which statements the zero input skips:

Where control goes after its handler:

Why the last valid count is still processed:

My post-run explanation and any correction:


### Check empty input and failures at different positions

Test every row using your complete unchanged ParcelMath class and caller. Change only the containerCounts initializer. Predict every output line before each run, record the actual lines, and explain any difference. Count Per container, Packed and Next input lines and check that Finished appears once. Explain why the empty case prints no Next input, why an initial zero must not prevent the next valid count, and why a final zero still reaches Finished. For two consecutive zero counts, explain why neither produces Packed while both receive a response. Repair any mismatch, repeat the affected cases and restore {2, 0, 5}. Finally explain which test would expose a catch around the whole loop.

| containerCounts | Predicted output | Actual output | Match or repair |
|---|---|---|---|
| {2, 0, 5} | | | |
| {} | | | |
| {0, 2} | | | |
| {5, 0} | | | |
| {0, 0} | | | |


Counts of successful reports, Packed and Next input for each case:

Why Finished appears once in every case:

What the empty case checks:

What the initial-zero and final-zero cases check separately:

Why the repeated-zero case reaches both handlers:

Which case exposes an outside-loop catch and why:

My correction and repeated checks:

My actual output after restoring the baseline:


<details>
<summary>Show answer</summary>

ParcelMath.perContainer performs integer division. The first input 2 returns 5, so Per container: 5 and Packed print before Next input. The zero input fails before the Per container print can finish its argument; Packed is skipped and the matching catch prints Choose a nonzero container count. Next input follows either path. The final input 5 returns 2 and prints the two success lines and progress marker. Finished runs after the loop. The class and all caller data are included, and the handler is inside each iteration so one invalid count does not discard later inputs. The empty array performs no loop work and prints only Finished. With {0, 2}, handling the first zero lets the valid second input produce 5 and Packed. With {5, 0}, the final caught failure still reaches Next input and then Finished. With {0, 0}, both failures get a response and progress marker, and no Packed line appears. These tests check handler placement and progress rather than only one successful quotient. An outside catch would exit the loop on its first failure and skip any later inputs.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {2, 0, 5};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Per container: 5
Packed
Next input
Choose a nonzero container count.
Next input
Per container: 2
Packed
Next input
Finished
```

Common error: Placing the handler around the entire loop and losing later inputs. Printing Packed from a failed path. Returning a made-up numeric result for division by zero. Putting Next input only in the successful try body. Changing fixed method names, input counts or output labels.

**Additional test: Empty input array.** With no inputs, there is no loop iteration, so there is no calculation, handler response or Next input. Finished still prints once.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Finished
```

**Additional test: Initial zero followed by a valid count.** The first failure receives its response and Next input. The valid second count still returns 5 and reaches Packed.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {0, 2};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Choose a nonzero container count.
Next input
Per container: 5
Packed
Next input
Finished
```

**Additional test: A valid count followed by a final zero.** The first count returns 2. The final failure still reaches its Next input, and Finished remains reachable after the loop.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {5, 0};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Per container: 2
Packed
Next input
Choose a nonzero container count.
Next input
Finished
```

**Additional test: Two consecutive zero counts.** Each failed input has its own response and progress marker. Neither reaches Packed, but the program still prints Finished once.

```java
class ParcelMath {
    public static int perContainer(int parcels, int containers) {
        return parcels / containers;
    }
}
int[] containerCounts = {0, 0};
for (int containers : containerCounts) {
    try {
        System.out.println("Per container: " + ParcelMath.perContainer(10, containers));
        System.out.println("Packed");
    } catch (ArithmeticException problem) {
        System.out.println("Choose a nonzero container count.");
    }
    System.out.println("Next input");
}
System.out.println("Finished");
```

Expected output:

```text
Choose a nonzero container count.
Next input
Choose a nonzero container count.
Next input
Finished
```

</details>


## Summary

A runtime exception carries a type and diagnostic information. A failure skips unfinished statements and searches outward for a matching handler. Handler placement determines whether the next input can still be processed. A stack trace connects the failing operation to its callers.

Close the answers. Explain the difference between resuming after a catch and resuming the failed statement. Trace share to quote to the outer handler.


## Reflection

A form processes several independent entries. Identify one failure that should reject only its current entry and one failure that should stop the entire operation. Explain where each handler belongs.

**My design and explanation:**

Next, you will raise your own exceptions and state which failures a method can pass to its caller.


## Supplemental Reading

- [Catching and handling Java exceptions](https://dev.java/learn/exceptions/catching-handling/) explains handler placement and recovery.
- [Java 21 Throwable API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Throwable.html) documents messages and stack-trace inspection.
- [Java 21 exception rules](https://docs.oracle.com/javase/specs/jls/se21/html/jls-11.html) defines exception control flow.
